# ZS601 2DGS：LiDAR B → C（L4）

按顺序运行以下 Cell。每一步都会在控制台打印当前阶段、GPU、Drive 输出目录和结果；C 组的冒烟测试与正式训练已拆开。

- **B 组**：LiDAR 初始化，`lambda_dist=0`，补跑 150k 后的渲染与指标。
- **C 组**：LiDAR 初始化，`lambda_dist=1000`，先冒烟测试，再正式训练至 150k。
- **预览**：每 5k 固定 10 个相机，输出 RGB、depth、normal、Gaussian ellipsoid。


## 1. 检查 L4 GPU 与 Python 环境

应看到 GPU 名称、显存、Python 和 PyTorch/CUDA 版本。


In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time

def banner(text):
    print("\n" + "=" * 88, flush=True)
    print(text, flush=True)
    print("=" * 88, flush=True)

banner("STEP 1/7  检查运行时")
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.9"
subprocess.run(["nvidia-smi"], check=True)
print("Python:", sys.version, flush=True)
try:
    import torch
    print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda, flush=True)
    print("CUDA available:", torch.cuda.is_available(), flush=True)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0), flush=True)
        print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), flush=True)
except Exception as exc:
    print("PyTorch check warning:", repr(exc), flush=True)


## 2. 挂载 Google Drive

训练日志、预览、断点和指标都会写入 Drive，不依赖 Colab 临时磁盘。


In [ ]:
from google.colab import drive

banner("STEP 2/7  挂载 Drive")
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive/LCCDataset/zs601_output")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("DRIVE_ROOT =", DRIVE_ROOT, flush=True)
print("Drive mounted:", DRIVE_ROOT.exists(), flush=True)


## 3. 同步 GitHub 代码并做语法检查

每次运行都会拉取分支 `2dgs-zs601-mask-init` 的最新提交，并在启动训练前校验启动脚本。


In [ ]:
banner("STEP 3/7  同步并校验 GitHub 代码")
REPO = Path("/content/ZS601_3DGS")
BRANCH = "2dgs-zs601-mask-init"
if not REPO.exists():
    subprocess.run([
        "git", "clone", "--recursive", "--branch", BRANCH,
        "https://github.com/VISjudy/ZS601_3DGS.git", str(REPO)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "submodule", "update", "--init", "--recursive"], check=True)

SOURCE = REPO / "2d-gaussian-splattingWithMask"
LAUNCHER = SOURCE / "colab/run_parallel_experiment.py"
subprocess.run([sys.executable, "-m", "py_compile", str(LAUNCHER)], check=True)
commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True).strip()
print("Git commit:", commit, flush=True)
print("Launcher syntax: OK", flush=True)
print("Launcher:", LAUNCHER, flush=True)


## 4. 检查 B/C 现有断点和预览

只读取状态，不启动训练。会显示最新运行目录、阶段、检查点和各类预览数量。


In [ ]:
banner("STEP 4/7  检查已有实验")
PREFIXES = {
    "b": "zs601_2dgs_B_lidar_parallel_",
    "c": "zs601_2dgs_C_lidar_distortion_parallel_",
}

def latest_run(mode):
    runs = [p for p in DRIVE_ROOT.glob(PREFIXES[mode] + "*") if p.is_dir()]
    return max(runs, key=lambda p: p.stat().st_mtime) if runs else None

def read_status(run_dir):
    try:
        return json.loads((run_dir / "status.json").read_text())
    except Exception:
        return {}

def checkpoints(run_dir):
    return sorted(
        (run_dir / "model").glob("chkpnt*.pth"),
        key=lambda p: int(p.stem.replace("chkpnt", "")),
    )

def preview_counts(run_dir):
    root = run_dir / "model" / "previews"
    return {
        "rgb": len(list(root.rglob("*rgb*.png"))),
        "depth": len(list(root.rglob("*depth*.png"))),
        "normal": len(list(root.rglob("*normal*.png"))),
        "gaussian": len(list(root.rglob("*gaussian*.png"))) + len(list(root.rglob("*ellipsoid*.png"))),
    }

def show_run(mode):
    run_dir = latest_run(mode)
    print(f"\n[{mode.upper()}] latest =", run_dir, flush=True)
    if run_dir is None:
        print("  尚无运行目录", flush=True)
        return None
    print("  status =", json.dumps(read_status(run_dir), ensure_ascii=False), flush=True)
    print("  checkpoints =", [p.name for p in checkpoints(run_dir)], flush=True)
    print("  preview files =", preview_counts(run_dir), flush=True)
    return run_dir

B_RUN = show_run("b")
C_RUN = show_run("c")


## 5. B 组：从断点补跑渲染、测试指标和 LiDAR 几何指标

B 组已训练到 150k。本 Cell 复用 `chkpnt150000.pth`，只补跑测试渲染、PSNR/SSIM/LPIPS 和几何评测，不会重训。静态评测已不再强制依赖 `mediapy`。


In [ ]:
banner("STEP 5/7  恢复并完成 B 组评测")

def run_live(args, label):
    print(f"[{label}] 命令开始；下面会持续输出日志。", flush=True)
    print(" ".join(str(x) for x in args), flush=True)
    started = time.time()
    rc = subprocess.call([str(x) for x in args], cwd=str(SOURCE))
    print(f"[{label}] RETURN_CODE={rc}  elapsed={((time.time()-started)/60):.1f} min", flush=True)
    if rc != 0:
        raise RuntimeError(f"{label} failed; see Drive logs above.")
    return rc

B_RUN = latest_run("b")
if B_RUN is None:
    raise FileNotFoundError("没有找到 B 组 Drive 目录")
b_status = read_status(B_RUN)
if b_status.get("stage") == "complete":
    print("B 已完成，跳过。", flush=True)
else:
    cp150 = B_RUN / "model/chkpnt150000.pth"
    if not cp150.exists():
        raise FileNotFoundError(f"B 组缺少 150k 断点：{cp150}")
    print("B_OUTPUT_DIR =", B_RUN, flush=True)
    print("B_CHECKPOINT =", cp150, flush=True)
    run_live([
        sys.executable, "-u", LAUNCHER, "--mode", "b",
        "--resume-output", B_RUN, "--postprocess-only"
    ], "B POSTPROCESS")

b_status = read_status(B_RUN)
print("B_FINAL_STATUS =", json.dumps(b_status, ensure_ascii=False), flush=True)
if b_status.get("stage") != "complete":
    raise RuntimeError("B 组评测尚未完成，暂停 C 组。")
summary = B_RUN / "results_summary.json"
if summary.exists():
    print(summary.read_text(), flush=True)


## 6A. C 组冒烟测试（distortion 开启）

独立运行 4k 轮冒烟测试，以确保 distortion 已进入有效阶段，并检查峰值显存不超过 13.5GB。成功后状态应为 `preflight_complete`。


In [ ]:
banner("STEP 6A/7  C 组冒烟测试")
C_RUN = latest_run("c")
c_status = read_status(C_RUN) if C_RUN else {}
c_cps = checkpoints(C_RUN) if C_RUN else []

if c_status.get("stage") in ("preflight_complete", "training", "rendering", "complete") or c_cps:
    print("C 已有可继续状态，跳过重复冒烟测试。", flush=True)
    print("C_OUTPUT_DIR =", C_RUN, flush=True)
    print("C_STATUS =", json.dumps(c_status, ensure_ascii=False), flush=True)
else:
    run_live([sys.executable, "-u", LAUNCHER, "--mode", "c", "--preflight-only"], "C SMOKE")
    C_RUN = latest_run("c")
    c_status = read_status(C_RUN)
    print("C_OUTPUT_DIR =", C_RUN, flush=True)
    print("C_SMOKE_STATUS =", json.dumps(c_status, ensure_ascii=False), flush=True)
    if c_status.get("stage") != "preflight_complete":
        raise RuntimeError("C 组冒烟测试未通过；不要运行正式训练 Cell。")


## 6B. C 组正式训练（150k）

此 Cell 会持续打印 loss、distortion、normal loss、Gaussian 数量、速度、GPU 当前/峰值显存和输出目录。若已有断点则自动恢复；否则复用刚才的冒烟测试目录开始正式训练。训练后自动渲染并计算指标。


In [ ]:
banner("STEP 6B/7  C 组正式训练")
C_RUN = latest_run("c")
if C_RUN is None:
    raise FileNotFoundError("请先运行 6A 冒烟测试 Cell。")
c_status = read_status(C_RUN)
c_cps = checkpoints(C_RUN)
print("C_OUTPUT_DIR =", C_RUN, flush=True)
print("C_START_STATUS =", json.dumps(c_status, ensure_ascii=False), flush=True)
print("C_CHECKPOINTS =", [p.name for p in c_cps], flush=True)

if c_status.get("stage") == "complete":
    print("C 已完成，跳过正式训练。", flush=True)
elif c_cps:
    run_live([
        sys.executable, "-u", LAUNCHER, "--mode", "c",
        "--resume-output", C_RUN, "--resume-training"
    ], "C RESUME FORMAL")
elif c_status.get("stage") == "preflight_complete":
    run_live([
        sys.executable, "-u", LAUNCHER, "--mode", "c",
        "--resume-output", C_RUN, "--skip-preflight"
    ], "C FORMAL")
else:
    raise RuntimeError("C 状态不可安全恢复；先查看上方 status，避免覆盖现场。")

c_status = read_status(C_RUN)
print("C_FINAL_STATUS =", json.dumps(c_status, ensure_ascii=False), flush=True)
if c_status.get("stage") != "complete":
    raise RuntimeError("C 组未完成，请保留本 Cell 输出用于诊断。")


## 7. 最终检查与结果汇总

确认 B/C 的最终状态、检查点、指标文件，以及每 5k 固定 10 视角的 RGB/depth/normal/Gaussian 预览。


In [ ]:
banner("STEP 7/7  最终检查")
for mode in ("b", "c"):
    run_dir = latest_run(mode)
    print(f"\n===== {mode.upper()} =====", flush=True)
    print("OUTPUT_DIR =", run_dir, flush=True)
    if run_dir is None:
        continue
    print("STATUS =", json.dumps(read_status(run_dir), ensure_ascii=False), flush=True)
    print("CHECKPOINTS =", [p.name for p in checkpoints(run_dir)], flush=True)
    print("PREVIEW_FILES =", preview_counts(run_dir), flush=True)
    for name in ("results_summary.json", "geometry_metrics.json"):
        p = run_dir / name
        print(name, "exists =", p.exists(), flush=True)
        if p.exists():
            print(p.read_text(), flush=True)
print("\n检查完成。Drive 根目录：", DRIVE_ROOT, flush=True)
